In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [21]:
df = pd.read_csv("fmnist_small.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [22]:
df.shape

(6000, 785)

In [30]:
x = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [31]:
xTrain, xTest, yTrain, yTest = train_test_split(x, y, test_size=0.3)

In [32]:
xTrain = xTrain/255.00
xTest = xTest/255.00

In [33]:
xTrain

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(4200, 784))

In [34]:
import torch
from torch.utils.data import Dataset, DataLoader

In [35]:
class CustomData(Dataset):

    def __init__(self, features, label):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.label = torch.tensor(label, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.label[index]

In [36]:
trainData = CustomData(xTrain, yTrain)

In [37]:
testData = CustomData(xTest, yTest)

In [38]:
trainDataLoader = DataLoader(trainData, batch_size=64, shuffle=True)
testDataLoader = DataLoader(testData, batch_size=64, shuffle=False)

In [39]:
import torch.nn as nn

In [76]:
class NeuralNetwork(nn.Module):

    def __init__(self, feature_len):
        super().__init__()

        self.linear = nn.Sequential(
            nn.Linear(feature_len, 128, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 10)
        )

    def forward(self, features):
        predict = self.linear(features)
        return predict

In [77]:
epochs = 50
learning_rate = 0.01

In [78]:
model = NeuralNetwork(xTrain.shape[1])
loss_fn = nn.CrossEntropyLoss()
optim = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [79]:
for epoch in range(epochs):
    for feature, label in trainDataLoader:
        y_pred = model(feature)
        loss = loss_fn(y_pred, label)
        optim.zero_grad()
        loss.backward()
        optim.step()
    print(f"Epochs: {epoch + 1}, Loss: {loss.item()}")

Epochs: 1, Loss: 1.3338606357574463
Epochs: 2, Loss: 1.072874665260315
Epochs: 3, Loss: 0.9178393483161926
Epochs: 4, Loss: 0.9008307456970215
Epochs: 5, Loss: 0.8117974400520325
Epochs: 6, Loss: 0.7353312969207764
Epochs: 7, Loss: 0.5958341956138611
Epochs: 8, Loss: 0.5766853094100952
Epochs: 9, Loss: 0.7569977641105652
Epochs: 10, Loss: 0.5960876941680908
Epochs: 11, Loss: 0.43688637018203735
Epochs: 12, Loss: 0.4855687618255615
Epochs: 13, Loss: 0.36701372265815735
Epochs: 14, Loss: 0.33859968185424805
Epochs: 15, Loss: 0.46447357535362244
Epochs: 16, Loss: 0.2846418023109436
Epochs: 17, Loss: 0.23740394413471222
Epochs: 18, Loss: 0.3171953856945038
Epochs: 19, Loss: 0.33071082830429077
Epochs: 20, Loss: 0.17129139602184296
Epochs: 21, Loss: 0.33967024087905884
Epochs: 22, Loss: 0.3841326832771301
Epochs: 23, Loss: 0.29978322982788086
Epochs: 24, Loss: 0.2573520243167877
Epochs: 25, Loss: 0.23753872513771057
Epochs: 26, Loss: 0.1511247605085373
Epochs: 27, Loss: 0.16821686923503876


In [80]:
model.eval()

NeuralNetwork(
  (linear): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=False)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=32, bias=True)
    (4): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=10, bias=True)
  )
)

In [70]:
x = torch.randn(3, 784)

In [71]:
x.shape

torch.Size([3, 784])

In [72]:
with torch.no_grad():
    output = model(x)

In [73]:
output

tensor([[-9.0169, -4.8974,  8.5070,  1.8689,  1.5981, 10.5072, -1.4122, -0.9740,
         -2.7947, -7.2798],
        [ 2.4286, -1.7804,  1.3239,  4.1688, -5.5828, 10.0204, -4.8273,  0.8804,
         -8.1443, -1.2368],
        [-4.4408,  0.0316,  3.3365,  8.5617,  5.4843,  3.9098, -8.7895, -2.2933,
         -5.5735, -7.6275]])

In [74]:
torch.max(output, 1)

torch.return_types.max(
values=tensor([10.5072, 10.0204,  8.5617]),
indices=tensor([5, 5, 3]))

In [81]:
total = 0
correct = 0

with torch.no_grad():
    for feature, label in testDataLoader:
        model_output = model(feature)
        _, prediction = torch.max(model_output, 1)
        total += label.shape[0]
        correct += (prediction == label).sum().item()
print(correct/total)

0.7883333333333333
